In [1]:
import torch
import torch.nn as nn
import torch.optim as optim


In [2]:
sentence = "I am going to school"

In [3]:
words = sentence.lower().split()
vocab = list(set(words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

In [8]:
# Convert words to indices
data = [word2idx[w] for w in words]

# Create input-target pairs for next-word prediction
# Input: word t, Target: word t+1
X = data[:-1]  # [I, am, going, to]
Y = data[1:]   # [am, going, to, school]

X = torch.tensor(X).unsqueeze(1)  # shape: [seq_len, batch=1]
Y = torch.tensor(Y)               # shape: [seq_len]

vocab_size = len(vocab)
embed_size = 8
hidden_size = 16
seq_len = X.shape[0]

# -----------------------------
# 2️⃣ Define RNN Model
# -----------------------------
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(SimpleRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(input_size=embed_size, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, h):
        x = self.embedding(x)         # Convert word idx to embedding
        out, h = self.rnn(x, h)      # RNN forward
        out = self.fc(out.squeeze(1)) # Predict next word
        return out, h

# -----------------------------
# 3️⃣ Initialize Model, Loss, Optimizer
# -----------------------------
model = SimpleRNN(vocab_size, embed_size, hidden_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# -----------------------------
# 4️⃣ Training Loop
# -----------------------------
epochs = 200
for epoch in range(epochs):
    h = torch.zeros(1, 1, hidden_size)  # Initialize hidden state
    optimizer.zero_grad()
    
    loss_total = 0
    for t in range(seq_len):
        x_t = X[t].unsqueeze(0)  # current word
        y_t = Y[t].unsqueeze(0)  # next word
        
        output, h = model(x_t, h)
        loss = criterion(output, y_t)
        loss.backward(retain_graph=True)  # BPTT
        loss_total += loss.item()
    
    optimizer.step()
    
    if (epoch+1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss_total:.4f}")

# -----------------------------
# 5️⃣ Test Prediction
# -----------------------------
h = torch.zeros(1, 1, hidden_size)
word = "i"  # starting word
print("Prediction sequence:")

for _ in range(5):
    x = torch.tensor([word2idx[word]]).unsqueeze(0)
    out, h = model(x, h)
    pred_idx = out.argmax(dim=1).item()
    pred_word = idx2word[pred_idx]
    print(pred_word, end=" ")
    word = pred_word




Epoch 50, Loss: 0.0406
Epoch 100, Loss: 0.0173
Epoch 150, Loss: 0.0106
Epoch 200, Loss: 0.0073
Prediction sequence:
am going to school am 

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim


In [10]:
sentence = "i have completed machine learning".split()

vocab = list(set(sentence))
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for word, i in word2idx.items()}

vocab_size = len(vocab)
print(word2idx)


{'i': 0, 'learning': 1, 'completed': 2, 'have': 3, 'machine': 4}


In [11]:
inputs = []
targets = []

for i in range(len(sentence) - 1):
    inputs.append(word2idx[sentence[i]])
    targets.append(word2idx[sentence[i+1]])

inputs = torch.tensor(inputs)
targets = torch.tensor(targets)


In [12]:
embedding_dim = 5
hidden_size = 5


In [13]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)          # (batch, seq_len, embed_dim)
        h0 = torch.zeros(1, x.size(0), hidden_size)
        out, _ = self.rnn(x, h0)        # hidden states
        out = self.fc(out[:, -1, :])    # last time step
        return out


In [14]:
model = RNNModel(vocab_size, embedding_dim, hidden_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


In [15]:
epochs = 300

for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(inputs.unsqueeze(0))  # add batch dim
    loss = criterion(outputs, targets[-1].unsqueeze(0))

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 1.6516
Epoch 50, Loss: 0.0473
Epoch 100, Loss: 0.0180
Epoch 150, Loss: 0.0107
Epoch 200, Loss: 0.0073
Epoch 250, Loss: 0.0053


In [16]:
with torch.no_grad():
    output = model(inputs.unsqueeze(0))
    predicted_idx = torch.argmax(output, dim=1).item()

print("Predicted next word:", idx2word[predicted_idx])


Predicted next word: learning


In [17]:
eng_sentence = "i have completed machine learning".split()
fr_sentence  = "j ai terminé apprentissage machine".split()


In [18]:
fr_sentence = ["<SOS>"] + fr_sentence + ["<EOS>"]


In [19]:
eng_vocab = list(set(eng_sentence))
fr_vocab  = list(set(fr_sentence))

eng_word2idx = {w:i for i,w in enumerate(eng_vocab)}
fr_word2idx  = {w:i for i,w in enumerate(fr_vocab)}
fr_idx2word  = {i:w for w,i in fr_word2idx.items()}


In [20]:
encoder_input = torch.tensor([[eng_word2idx[w] for w in eng_sentence]])
decoder_input = torch.tensor([[fr_word2idx[w] for w in fr_sentence[:-1]]])
decoder_target = torch.tensor([[fr_word2idx[w] for w in fr_sentence[1:]]])


In [21]:
embedding_dim = 8
hidden_size = 16


In [22]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embedding(x)
        _, hidden = self.rnn(x)
        return hidden


In [23]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x)
        output, hidden = self.rnn(x, hidden)
        output = self.fc(output)
        return output, hidden


In [24]:
encoder = Encoder(len(eng_vocab), embedding_dim, hidden_size)
decoder = Decoder(len(fr_vocab), embedding_dim, hidden_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()), lr=0.01
)


In [25]:
epochs = 500

for epoch in range(epochs):
    optimizer.zero_grad()

    # Encoder
    hidden = encoder(encoder_input)

    # Decoder (teacher forcing)
    output, _ = decoder(decoder_input, hidden)

    loss = criterion(
        output.reshape(-1, len(fr_vocab)),
        decoder_target.reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 1.8811
Epoch 100, Loss: 0.0067
Epoch 200, Loss: 0.0026
Epoch 300, Loss: 0.0014
Epoch 400, Loss: 0.0009


In [26]:
with torch.no_grad():
    hidden = encoder(encoder_input)
    decoder_input = torch.tensor([[fr_word2idx["<SOS>"]]])

    translated = []

    for _ in range(10):
        output, hidden = decoder(decoder_input, hidden)
        pred_idx = output.argmax(dim=2).item()
        word = fr_idx2word[pred_idx]

        if word == "<EOS>":
            break

        translated.append(word)
        decoder_input = torch.tensor([[pred_idx]])

print("Translated sentence:", " ".join(translated))


Translated sentence: j ai terminé apprentissage machine
